##### 코랩을 사용할 경우

In [ ]:
try:
    # Google Drive를 Colab(/content/drive)에 마운트
    from google.colab import drive
    drive.mount('/google_drive')

    # Colab에서 작업 디렉토리로 이동
    %cd /google_drive/Othercomputers/'내 컴퓨터'/sec05
    %ls
except ImportError:
    pass

##### 이전 노트북 실행

In [1]:
%%capture
get_ipython().run_line_magic("run", "01_data_preprocessing.ipynb")
get_ipython().run_line_magic("run", "02_model_definition.ipynb")

##### 임포트

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy

##### Device 설정

In [3]:
# GPU(CUDA) 사용 가능 시 'cuda', 아니면 'cpu' 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")

사용 장치: cpu


##### 옵티마이저 생성

In [4]:
# 손실 함수(CrossEntropyLoss) 생성
loss_fn = nn.CrossEntropyLoss()

# Adam 옵티마이저(Optimizer) 생성
# lr=0.001: 학습률 (모든 파라미터(가중치, 편향)를 얼마나 크게 변경할지)
optimizer = optim.Adam(model.parameters(), lr=0.001)

##### 학습하기 (학습률 스케쥴러 적용)

In [5]:
# 학습 에포크 및 배치 크기 설정
epochs     = 300
batch_size = 16

# 조기 종료(Early Stopping) 설정 #####################################
# patience: 검증 손실이 개선되지 않아도 허용할 에포크 수
# patience_counter: 개선 없이 누적된 에포크 수
# best_val_loss: 현재까지 가장 낮은 검증 손실 (초기값: 무한대)
# best_model_state: 검증 손실이 가장 낮았을 때의 모델 파라미터 복사본
patience         = 10
patience_counter = 0
best_val_loss    = float('inf') # 양의 무한대로 초기화
best_model_state = None
####################################################################

# 학습률 스케쥴러(Learning Rate Scheduler) 생성 #######################
# ReduceLROnPlateau: 검증 손실이 개선되지 않으면 학습률을 자동으로 줄임
#   mode='min'   : 'min': 낮을수록 좋음 (검증 손실 기준) / 'max': 높을수록 좋음 (검증 정확도 기준)
#   factor=0.5   : 개선 없을 때 lr = lr * 0.5 (절반으로 줄임)
#   patience=3   : 3 에포크 동안 개선 없으면 lr 감소
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)
###################################################################

# 에포크 반복
for epoch in range(epochs):
    # 훈련 단계 -----------------------------------------------------
    # 학습 모드로 설정: 파라미터 업데이트 및 드롭아웃 활성화
    model.train()

    # 훈련 손실과 정확도 누적 변수 초기화
    batch_loss_sum   = 0
    batch_correct_sum = 0

    for i in range(0, len(train_x_tensor), batch_size):
        batch_x_tensor = train_x_tensor[i:i + batch_size]
        batch_y_tensor = train_y_tensor[i:i + batch_size]

        # 1. 그래디언트 초기화
        optimizer.zero_grad()

        # 2. 모델 출력 얻기
        train_outputs = model(batch_x_tensor)

        # 3. 손실 계산
        loss = loss_fn(train_outputs, batch_y_tensor)

        # 4. 역전파: 그래디언트 계산
        loss.backward()

        # 5. 가중치 업데이트
        optimizer.step()

        # 각 배치 손실의 합산
        batch_loss_sum  += loss.item()

        # 각 배치에서 맞춘 개수 합산: sum(): True(=1) 개수 합산
        batch_correct_sum += (train_outputs.argmax(dim=1) == batch_y_tensor).sum().item()

    # 훈련 평균 손실: 각 배치 손실의 합산 / 배치 개수
    epoch_loss = batch_loss_sum / (len(train_x_tensor) / batch_size)

    # 훈련 정확도: 맞춘 개수 / 전체 개수 * 100
    epoch_acc = batch_correct_sum / len(train_x_tensor) * 100

    # 검증 단계 -----------------------------------------------------
    # 평가 모드로 설정: 드롭아웃 비활성화
    model.eval()

    # 검증 손실과 정확도 누적 변수 초기화
    val_loss    = 0
    val_correct = 0

    # 역전파/기울기 계산없이 검증셋으로 예측 수행
    with torch.inference_mode():  # torch.no_grad():
        # 검증셋 예측
        val_outputs  = model(val_x_tensor)
        # 검증 손실 계산
        val_loss     = loss_fn(val_outputs, val_y_tensor).item()
        # 검증 정확도 계산
        val_correct  = (val_outputs.argmax(dim=1) == val_y_tensor).sum().item()
        val_acc = val_correct / len(val_x_tensor) * 100

    # 스케줄러 적용 ###################################################
    # 수정전 학습률 가져오기
    prev_lr = optimizer.param_groups[0]['lr']
    # 검증 손실을 넘겨 개선 여부를 판단하고, 필요 시 lr을 줄임
    lr_scheduler.step(val_loss)
    # 수정된 학습률 가져오기
    current_lr = optimizer.param_groups[0]['lr']
    #################################################################

    if current_lr != prev_lr:
        # 출력하기 -----------------------------------------------------
        # 에포크마다 훈련·검증 지표를 나란히 출력하여 과적합 여부 확인
        print(f"Epoch [{epoch+1:3d}/{epochs}]  "
                f"훈련 손실: {epoch_loss:.4f}  훈련 정확도: {epoch_acc:5.1f}%  "
                f"검증 손실: {val_loss:.4f}  검증 정확도: {val_acc:5.1f}%  "
                f"lr: {current_lr:.6f}", end="")

        # 조기 종료 판단
        if val_loss < best_val_loss:
            # 검증 손실이 개선된 경우: 최적 상태 갱신
            best_val_loss    = val_loss
            patience_counter = 0
            # 현재 에포크의 모델 파라미터를 별도 메모리에 완전히 복사
            best_model_state = copy.deepcopy(model.state_dict())
            print("  ✓ 최적 모델 저장")
        else:
            # 검증 손실이 개선되지 않은 경우: 카운터 증가
            patience_counter += 1
            print(f"  (개선 없음 {patience_counter}/{patience})")
            if patience_counter >= patience:
                print(f"\n조기 종료: {patience} 에포크 동안 검증 손실 개선 없음")
                break

# 가장 성능이 좋았던 에포크의 가중치로 모델 복원
model.load_state_dict(best_model_state)

print(f"학습 완료  (최적 검증 손실: {best_val_loss:.4f})")


Epoch [ 23/300]  훈련 손실: 0.0602  훈련 정확도:  98.6%  검증 손실: 0.1003  검증 정확도:  94.4%  lr: 0.000500  ✓ 최적 모델 저장
Epoch [ 27/300]  훈련 손실: 0.0375  훈련 정확도: 100.0%  검증 손실: 0.1007  검증 정확도:  94.4%  lr: 0.000250  (개선 없음 1/10)
Epoch [ 31/300]  훈련 손실: 0.0464  훈련 정확도: 100.0%  검증 손실: 0.0983  검증 정확도:  94.4%  lr: 0.000125  ✓ 최적 모델 저장
Epoch [ 35/300]  훈련 손실: 0.0363  훈련 정확도:  99.3%  검증 손실: 0.0996  검증 정확도:  94.4%  lr: 0.000063  (개선 없음 1/10)
Epoch [ 39/300]  훈련 손실: 0.0401  훈련 정확도:  99.3%  검증 손실: 0.1020  검증 정확도:  94.4%  lr: 0.000031  (개선 없음 2/10)
Epoch [ 43/300]  훈련 손실: 0.0268  훈련 정확도: 100.0%  검증 손실: 0.1027  검증 정확도:  94.4%  lr: 0.000016  (개선 없음 3/10)
Epoch [ 47/300]  훈련 손실: 0.0401  훈련 정확도:  99.3%  검증 손실: 0.1025  검증 정확도:  94.4%  lr: 0.000008  (개선 없음 4/10)
Epoch [ 51/300]  훈련 손실: 0.0298  훈련 정확도: 100.0%  검증 손실: 0.1025  검증 정확도:  94.4%  lr: 0.000004  (개선 없음 5/10)
Epoch [ 55/300]  훈련 손실: 0.0416  훈련 정확도:  99.3%  검증 손실: 0.1026  검증 정확도:  94.4%  lr: 0.000002  (개선 없음 6/10)
Epoch [ 59/300]  훈련 손실: 0.0284  훈련 정확도: 100.0%  검증